# Chapter 08 Companion Notebook: SVR: Housing Price Regression

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch08_SVR_Housing_Price.ipynb)

This notebook accompanies Chapter 08 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Housing price: SVR

### Use "Housing price.csv".
- id: Unique identifier for each property  
- list: Date of property listing (YYYYMMDD)
- yr_built: The year the property was originally built.  
- yr_renovated: The year the property was last renovated (0 indicates no renovation).  
- price: Property price
- bedrooms: Number of bedrooms  
- bathrooms: Number of bathrooms  
- sqft_living: Living area size in square feet  
- sqft_lot: Lot size in square feet  
- floors: Number of floors  
- waterfront: 1 if property has waterfront view
- view: Quality level of property view (0 to 4)  
- condition: the overall condition of the property (1 to 5).  
- grade: the construction and design quality of the property (1-13).  

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv('Housing price.csv')
df.head()

### 1. Create the following variables: 'years since built' and 'years since renovated'.

In [ ]:
from datetime import datetime

# Convert the "date" column to a datetime format
df['list'] = pd.to_datetime(df['list'], format='%Y%m%d')

# Extract the "list year" from the date column
df['list_year'] = df['list'].dt.year

# Calculate "years since renovated" and "years since built"
df['years_since_renovated'] = df.apply(lambda row: row['list_year'] - row['yr_renovated']
            if row['yr_renovated'] > 0 else row['list_year'] - row['yr_built'], axis=1)
df['years_since_built'] = df['list_year'] - df['yr_built']
df.head()

### 2. Define dependent and independent variables. Divide the data into 75% training and 25% test set (use random_state=15). Then, scale the independent variables.
- Dependent variable: price
- Independent variables: 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'years_since_renovated', 'years_since_built'

In [ ]:
# Split the data into features and target variable
y = df.price
x = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront',
        'view', 'condition', 'grade', 'years_since_renovated', 'years_since_built']]

# Splitting the dataset into the Training set and Test set
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=15)

- `x = df.drop('value', axis=1)` creates a new DataFrame `x` by dropping the column 'value'.
    - `axis=1` indicates that the operation should be performed along columns (i.e., dropping a column).
- `y = df.value` assigns the values from the 'value' column to a variable `y`.
- `xtrain, xtest, ytrain, ytest = train_test_split()` splits the data into training and testing sets.

In [ ]:
# Feature scaling
scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain)
xtest = scaler.transform(xtest)

`scaler = StandardScaler()` creates a `StandardScaler` object. At this point, no calculations have been performed.

`xtrain = scaler.fit_transform(xtrain)` computes the mean and standard deviation of each feature using only the training data, then standardizes the training data by subtracting the mean and dividing by the standard deviation. The `fit_transform()` method combines two operations: `fit()` learns the scaling parameters (the mean and standard deviation), and `transform()` applies those parameters to the training data.

`xtest = scaler.transform(xtest)` standardizes the test data using the same mean and standard deviation computed from the training data. Only `transform()` is used because the test data should not influence the scaling parameters. Using the training statistics ensures that both the training and test data are on the same scale and prevents information from the test set from leaking into the training process.

In [ ]:
# SVM Regressor with linear kernel

svm1 = SVR(kernel='linear').fit(xtrain, ytrain)
pred1 = svm1.predict(xtest)
mse1 = mean_squared_error(ytest, pred1)

- `svm1 = SVR(kernel='linear').fit(xtrain, ytrain)` creates an SVR (Support Vector Regressor) object with a linear kernel.

In [ ]:
# SVM Regressor with polynomial kernel

svm2 = SVR(kernel='poly', degree=3).fit(xtrain, ytrain)
pred2 = svm2.predict(xtest)
mse2 = mean_squared_error(ytest, pred2)

- `svm2 = SVR(kernel='poly', degree=3).fit(xtrain, ytrain)` creates an SVR (Support Vector Regressor) object with a polynomial kernel of degree 3.

In [ ]:
# SVM Regressor with RBF kernel

svm3 = SVR(kernel='rbf').fit(xtrain, ytrain)
pred3 = svm3.predict(xtest)
mse3 = mean_squared_error(ytest, pred3)

- `svm3 = SVR(kernel='rbf').fit(xtrain, ytrain)` creates an SVR (Support Vector Regressor) object with an RBF kernel.

In [ ]:
print("SVM Linear MSE:", mse1)
print("SVM Polynomial MSE:", mse2)
print("SVM RBF MSE:", mse3)

### Hyperparameter tuning

In [ ]:
# Define parameter grid
param = {'C': [0.1, 1, 10, 100, 1000]}

# Create the SVR model with RBF kernel
svr = SVR(kernel='rbf')

# Grid search with cross-validation
search = GridSearchCV(svr, param, cv=5, scoring='neg_mean_squared_error').fit(xtrain, ytrain)

# Best parameters and best score
print("Best parameter:", search.best_params_)
print("Best MSE:", -search.best_score_)

- `param` specifies the hyperparameters to be tuned and the range of values to be searched over.
   - `C`: Regularization parameter.
- `svr = SVR(kernel='rbf')`: An SVR model with a radial basis function (RBF) kernel is instantiated.
- `GridSearchCV` exhaustively searches through a specified grid of hyperparameters, evaluating the model performance using cross-validation on each combination of hyperparameters.
   - `estimator`: the SVR model with RBF kernel (`svr`).
   - `cv`: The number of folds for cross-validation, set to 5 (`cv=5`).
   - `scoring`: The evaluation metric used to select the best parameters (negative mean squared error).
- After fitting the grid search model, the best combination of hyperparameters (`best_params_`) and the corresponding best score in terms of negative mean squared error (`best_score_`) are printed.
    - `-` sign is used to obtain MSE